# Tomahawk 16 T CAD placement and combined-field input audit

Attach `Tomahawk_G1_G2_All_Inputs.zip` as a Kaggle dataset and run all cells.
The input finder accepts the ZIP, a renamed ZIP, or an extracted package folder
under `/kaggle/input`. For a local run, set `TOMAHAWK_INPUT_PATH` to the ZIP or
extracted folder.

**Names and scope.** The input package uses “G1” for its CAD/placement stage
and “G2” for its combined-field stage. These stage names are not claims of
geometric G1 or G2 surface continuity. This notebook verifies file hashes,
selected CAD boundary identifiers and BREP bytes, placement-table counts and
TF centerline closure, combined MGRID structure and axis value, and a
recomputed vacuum-flux seed. Its output is an **input audit**.

The source `G1_16T_validation_report.json` records a historical formal status
of `UNRESOLVED_PENDING_INTERFACE_REVIEW`; this notebook does not rerun or
clear that interface gate. It does not evaluate laser ray transport, thermal
pulses, wall capture, or the FreeCAD `TF_Sector` solid geometry. A laser-port
count is a table check only. FreeCAD assembly is a separate task performed by
`RUN_G1_G2_FROM_ONE_ZIP.py`.


In [ ]:
import collections
import csv
import hashlib
import io
import json
import math
import os
from pathlib import Path
import zipfile
import lzma

import numpy as np
from scipy.io import netcdf_file
from numpy.polynomial.legendre import leggauss

PACKAGE = "Tomahawk_G1_G2_All_Inputs"
CAD = "Tomahawk_TF48_16FP_16T_CAD_Handoff"
G2 = "Tomahawk_G2_16T_Preflight"
ZIP_NAME = PACKAGE + ".zip"
MANIFEST_NAME = "MANIFEST_SHA256.json"
REQUIRED = {CAD + "/Tomahawk_Liquid_Wall_SOURCE.FCStd",
            CAD + "/G1_16T_validation_report.json",
            G2 + "/G2_field_geometry_audit.json",
            G2 + "/mgrid_tomahawk_16T_TF_PF_16FP.nc.xz"}

class FolderBundle:
    def __init__(self, root):
        self.root = root
    def read(self, relative):
        return (self.root / relative).read_bytes()
    def close(self):
        pass

def valid_manifest(data):
    try:
        candidate = json.loads(data)
        if (isinstance(candidate, dict) and REQUIRED <= candidate.keys()
                and all(isinstance(name, str) and isinstance(digest, str)
                        and len(digest) == 64 for name, digest in candidate.items())):
            return candidate
    except (ValueError, UnicodeDecodeError, TypeError):
        pass
    return None

def open_package(path):
    if path.is_dir():
        manifests = sorted(path.rglob(MANIFEST_NAME))
        for manifest_file in manifests:
            found = valid_manifest(manifest_file.read_bytes())
            if found is not None and all((manifest_file.parent / name).is_file() for name in found):
                return FolderBundle(manifest_file.parent), "", found, manifest_file.parent
    elif path.is_file() and path.suffix.lower() == ".zip":
        try:
            archive = zipfile.ZipFile(path)
        except (OSError, zipfile.BadZipFile):
            return None
        try:
            names = set(archive.namelist())
            manifests = sorted(name for name in names
                               if name == MANIFEST_NAME or name.endswith("/" + MANIFEST_NAME))
            for member in manifests:
                found = valid_manifest(archive.read(member))
                prefix = member[:-len(MANIFEST_NAME)]
                if found is not None and all(prefix + name in names for name in found):
                    return archive, prefix, found, path
        except (OSError, zipfile.BadZipFile):
            pass
        archive.close()
    return None

# Kaggle commonly mounts dataset ZIP contents as an extracted directory.
# Search mounted inputs first, then local files; the explicit override takes priority.
override = os.environ.get("TOMAHAWK_INPUT_PATH") or os.environ.get("TOMAHAWK_G1_G2_ZIP")
roots = [Path(override).expanduser()] if override else []
roots += [Path("/kaggle/input"), Path.cwd()]
selected = None
seen = set()
for root in roots:
    if not root.exists():
        continue
    candidates = [root] if root.is_file() else []
    if root.is_dir():
        candidates += [root]
        candidates += sorted(p for p in root.rglob("*") if p.is_file() and p.suffix.lower() == ".zip")
    for path in candidates:
        resolved = path.resolve()
        if resolved in seen:
            continue
        seen.add(resolved)
        selected = open_package(path)
        if selected is not None:
            break
    if selected is not None:
        break

if selected is None:
    mounted = Path("/kaggle/input")
    visible = ([str(p.relative_to(mounted)) for p in mounted.rglob("*") if p.is_file()][:30]
               if mounted.exists() else [])
    raise FileNotFoundError(
        "Tomahawk CAD/field inputs were not found. Attach the dataset containing " + ZIP_NAME +
        " or its extracted folder, then run again. Files under /kaggle/input: " +
        (", ".join(visible) if visible else "(none)"))
bundle, prefix, manifest, bundle_path = selected
outdir = Path(os.environ.get("TOMAHAWK_OUTPUT_DIR",
                             "/kaggle/working" if Path("/kaggle/working").exists()
                             else "./tomahawk_results"))
outdir.mkdir(parents=True, exist_ok=True)

def blob(relative):
    data = bundle.read(prefix + relative)
    actual = hashlib.sha256(data).hexdigest()
    assert actual == manifest[relative], "SHA-256 mismatch: " + relative
    return data

def records(relative):
    return list(csv.DictReader(io.StringIO(blob(relative).decode("utf-8-sig"))))

report = {
    "source_package": ZIP_NAME,
    "audit": "CAD_placement_and_combined_field_inputs",
    "status": "INPUT_CHECKS_PENDING",
    "scope": {
        "package_stage_labels": {"G1": "CAD and placement inputs", "G2": "combined-field inputs"},
        "surface_continuity_or_formal_interface_gate_evaluated": False,
        "not_assessed": ["laser ray transport", "thermal pulse response", "wall capture",
                         "FreeCAD TF_Sector solid geometry"],
    },
    "checks": {},
}
for relative in manifest:
    blob(relative)
report["checks"]["package_file_hashes"] = "PASS"
print("Selected:", bundle_path)
print("Verified", len(manifest), "package files against SHA-256 manifest")


## CAD source and placement tables (package stage G1)


In [ ]:
g1 = json.loads(blob(CAD + "/G1_16T_validation_report.json"))
audit = json.loads(blob(G2 + "/G2_field_geometry_audit.json"))
cad_bytes = blob(CAD + "/Tomahawk_Liquid_Wall_SOURCE.FCStd")
cad_sha = hashlib.sha256(cad_bytes).hexdigest()
assert cad_sha == g1["current_supplied_cad_boundary_check"]["source_fcstd_sha256"]
assert cad_sha == audit["CAD_boundary"]["cad_fcstd_sha256"]
with zipfile.ZipFile(io.BytesIO(cad_bytes)) as fcstd:
    members = set(fcstd.namelist())
    brep_name = "EV007_Plasma_Boundary.Shape.brp"
    assert brep_name in members and "Document.xml" in members
    brep_sha = hashlib.sha256(fcstd.read(brep_name)).hexdigest()
    document_xml = fcstd.read("Document.xml")
    assert b'name="EV007_Plasma_Boundary"' in document_xml
assert brep_sha == g1["current_supplied_cad_boundary_check"]["brep_sha256"]
assert brep_sha == audit["CAD_boundary"]["boundary_brep_sha256"]

tf = records(CAD + "/TF48_exact_centerlines_m.csv")
groups = collections.defaultdict(list)
for row in tf:
    groups[row["coil_id"]].append(row)
assert len(tf) == 48 * 512 and len(groups) == 48
closure_errors = []
for rows in groups.values():
    ordered = sorted(rows, key=lambda r: int(r["segment"]))
    assert len(ordered) == 512 and [int(r["segment"]) for r in ordered] == list(range(512))
    first = [float(ordered[0][key + "_m"]) for key in "xyz"]
    last = [float(ordered[-1]["next_" + key + "_m"]) for key in "xyz"]
    closure_errors.append(math.dist(first, last))
assert max(closure_errors) < 1e-7

pf = records(CAD + "/PF578_turn_placement.csv")
laser = records(CAD + "/laser16_ports_and_timing.csv")
rf = records(CAD + "/RF4_cassette_reserved_placement.csv")
assert len(pf) == 578 and len(set(r["coil_id"] for r in pf)) == 6
assert len(laser) == 16 and len(set(r["port_id"] for r in laser)) == 16
assert len(rf) == 4 and len(set(r["station_id"] for r in rf)) == 4
pf_currents = {name: {float(r["signed_current_A"]) for r in pf if r["coil_id"] == name}
               for name in set(r["coil_id"] for r in pf)}
assert all(len(values) == 1 for values in pf_currents.values())
for name, current in zip(audit["groups"][1:], audit["EXTCUR_A_per_turn_in_group_order"][1:]):
    assert math.isclose(next(iter(pf_currents[name])), current, rel_tol=1e-10, abs_tol=1e-6)

report["checks"]["CAD_source_and_placement_tables"] = "PASS"
report["source_boundary_report_status"] = {
    "historical_native_G1_formal_status": g1.get("historical_native_G1_formal_status")
}
report["CAD_placement_inputs"] = {"source_FCStd_sha256": cad_sha, "boundary_BREP_sha256": brep_sha,
                "TF_coils": len(groups), "TF_segments": len(tf), "TF_closure_max_m": max(closure_errors),
                "PF_turns": len(pf), "laser_ports": len(laser), "RF_stations": len(rf)}
print("CAD source and placement table checks: PASS", report["CAD_placement_inputs"])


## Combined MGRID input (package stage G2)


In [ ]:
compressed_relative = G2 + "/mgrid_tomahawk_16T_TF_PF_16FP.nc.xz"
compressed = blob(compressed_relative)
nc_path = outdir / "mgrid_tomahawk_16T_TF_PF_16FP.nc"
expected_nc = audit["combined_MGRID_sha256"]
def file_hash(path):
    digest = hashlib.sha256()
    with path.open("rb") as src:
        for chunk in iter(lambda: src.read(4 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()
if not nc_path.is_file() or file_hash(nc_path) != expected_nc:
    temp = nc_path.with_suffix(".partial")
    digest = hashlib.sha256()
    with lzma.LZMAFile(io.BytesIO(compressed)) as src, temp.open("wb") as dst:
        for chunk in iter(lambda: src.read(4 * 1024 * 1024), b""):
            dst.write(chunk)
            digest.update(chunk)
    assert digest.hexdigest() == expected_nc
    temp.replace(nc_path)
assert nc_path.stat().st_size == audit["combined_MGRID_bytes"]

nc = netcdf_file(str(nc_path), "r", mmap=False)
def scalar(name):
    return np.asarray(nc.variables[name].data).item()
assert (scalar("nfp"), scalar("kp"), scalar("nextcur")) == (16, 96, 7)
assert (scalar("ir"), scalar("jz")) == (201, 201)
assert nc.variables["mgrid_mode"].data.tobytes() == b"S"
currents = np.asarray(nc.variables["raw_coil_cur"].data, dtype=float)
assert np.allclose(currents, audit["EXTCUR_A_per_turn_in_group_order"], rtol=0, atol=1e-7)
labels = [row.tobytes().decode("ascii").strip() for row in nc.variables["coil_group"].data]
assert labels == audit["groups"]
for component in ("br", "bp", "bz"):
    tf_field = np.asarray(nc.variables[component + "_001"].data)
    assert np.isfinite(tf_field).all()
    assert np.array_equal(tf_field[:32], tf_field[32:64])
    assert np.array_equal(tf_field[:32], tf_field[64:96])
    for group in range(2, 8):
        pf_field = np.asarray(nc.variables[component + "_%03d" % group].data)
        assert np.isfinite(pf_field).all()
        assert np.array_equal(pf_field[0], pf_field[48])
        assert np.array_equal(pf_field[0], pf_field[95])
axis = {name: float(sum(currents[g-1] * nc.variables[key + "_%03d" % g].data[0,100,100]
                        for g in range(1,8)))
        for name,key in (("Br","br"),("Bphi","bp"),("Bz","bz"))}
assert abs(axis["Bphi"] - audit["axis_vacuum_field_phi0_R7p5_Z0_T"]["Bphi"]) < 1e-8
assert abs(axis["Bphi"] - 16.0) < 1e-5
report["checks"]["combined_MGRID_structure_and_axis"] = "PASS"
report["combined_field_input"] = {"combined_MGRID_sha256": expected_nc, "nfp": 16, "kp": 96,
                "groups": labels, "EXTCUR_A_per_turn": currents.tolist(), "axis_vacuum_field_T": axis}
print("Combined field grid input checks: PASS", axis)


## Fourier seed and vacuum-flux recomputation (package stage G2)


In [ ]:
flux_record = json.loads(blob(G2 + "/G2_vacuum_flux_seed.json"))
n_r, n_theta = 32, 128
x, w = leggauss(n_r)
rho, wr = (x + 1.0) / 2.0, w / 2.0
theta = np.arange(n_theta) * (2.0 * np.pi / n_theta)
rmin, rmax = float(scalar("rmin")), float(scalar("rmax"))
zmin, zmax = float(scalar("zmin")), float(scalar("zmax"))
dr = (rmax - rmin) / 200.0
dz = (zmax - zmin) / 200.0
flux_values = []
for plane in range(96):
    phi = 2.0 * np.pi * plane / (16 * 96)
    phase = theta - 16.0 * phi
    r = 7.5 + rho[:,None] * (3.0*np.cos(theta) + 0.01*np.cos(phase))
    z = rho[:,None] * (2.138969432449380*np.sin(theta) + 0.01*np.sin(phase))
    ri = (r - rmin) / dr
    zi = (z - zmin) / dz
    ir = np.clip(np.floor(ri).astype(int), 0, 199)
    iz = np.clip(np.floor(zi).astype(int), 0, 199)
    fr, fz = ri - ir, zi - iz
    bphi = np.zeros((201,201), dtype=float)
    for g in range(1,8):
        bphi += currents[g-1] * np.asarray(nc.variables["bp_%03d" % g].data[plane])
    values = ((1-fr)*(1-fz)*bphi[iz,ir] + fr*(1-fz)*bphi[iz,ir+1]
              + (1-fr)*fz*bphi[iz+1,ir] + fr*fz*bphi[iz+1,ir+1])
    jac = 3.0*2.138969432449380 + 0.01**2 + 0.01*(3.0+2.138969432449380)*np.cos(16*phi)
    flux_values.append(float(jac * (2*np.pi/n_theta) * np.sum((wr*rho)[:,None]*values)))
vacuum_flux_mean = float(np.mean(flux_values))
reference = flux_record["quadrature"][0]["mean_Wb"]
assert abs(vacuum_flux_mean-reference) < 1e-3

surface = audit["CAD_boundary"]["analytic_surface"]
assert audit["nfp"] == 16 and audit["CAD_boundary"]["unique_vertices"] == 6144
samples = outdir / "Fourier_boundary_seed_samples.csv"
with samples.open("w",newline="",encoding="utf-8") as stream:
    writer = csv.writer(stream)
    writer.writerow(("phi_rad","theta_rad","R_m","Z_m","X_m","Y_m"))
    for j in range(16):
        phi = 2*np.pi*j/16
        for k in range(64):
            t = 2*np.pi*k/64
            R = 7.5 + 3*np.cos(t) + 0.01*np.cos(t-16*phi)
            Z = 2.138969432449380*np.sin(t) + 0.01*np.sin(t-16*phi)
            writer.writerow((phi,t,R,Z,R*np.cos(phi),R*np.sin(phi)))
nc.close()
report["checks"]["Fourier_seed_and_vacuum_flux_recompute"] = "PASS"
report["combined_field_input"]["vacuum_flux_mean_Wb_32x128"] = vacuum_flux_mean
report["combined_field_input"]["vacuum_flux_reference_Wb_32x128"] = reference
report["combined_field_input"]["Fourier_boundary_sample_csv"] = samples.name
report["status"] = "PASS_CAD_PLACEMENT_AND_COMBINED_FIELD_INPUT_CHECKS"
print("Fourier seed and vacuum-flux recomputation: PASS",vacuum_flux_mean,"Wb")


In [ ]:
result = outdir / "Tomahawk_CAD_Field_Input_Audit.json"
result.write_text(json.dumps(report,indent=2)+"\n",encoding="utf-8")
result_zip = outdir / "Tomahawk_CAD_Field_Input_Audit_results.zip"
with zipfile.ZipFile(result_zip,"w",compression=zipfile.ZIP_DEFLATED) as output:
    output.write(result,arcname=result.name)
    output.write(samples,arcname=samples.name)
bundle.close()
print("Saved:", result)
print("Download Kaggle output:", result_zip)
